# Delete (undeploy) a MAAP DPS algorithm

The MAAP **Register Algorithm** GUI can register but **not** delete. Undeploying is a
`DELETE` against the OGC processes API, which is what this notebook does.

**When you need it**

- An algorithm was **renamed** (e.g. `umbra` / `umbra-ogc-test` →
  `disasters-umbra-process`). A rename registers a **NEW** process — the old one is
  left behind and must be deleted by hand, or the Submit Jobs dropdown keeps showing
  stale entries that still run old code.
- Throwaway `-ogc-test` experiments.
- Failed / partial registrations.

**Rules of the road**

- You delete by **`processID`** (an int MAAP assigns at registration), **not** by
  `name:version`. So: list first, then delete.
- **Only the deployer can delete their own process** — anything else returns `403`.
- Deletion removes the *process registration* only. Job history, outputs already in
  `nasa-disasters-staging`, and the built container are untouched.
- Re-registering the same name afterwards gets a **new** `processID`.

**Prereqs**

- Runs on the MAAP hub / ADE (needs MAAP auth). Kernel: the **`disasters_dps`** conda
  env — the same env `dps/*/build-env.sh` builds, which pins `maap-py`. If that kernel
  isn't in the picker:
  ```bash
  conda run -n disasters_dps pip install ipykernel
  conda run -n disasters_dps python -m ipykernel install --user --name disasters_dps \
      --display-name "Python 3 (disasters_dps)"
  ```
- A valid MAAP token. `MAAP()` reads it from the workspace; if `token` comes back empty,
  set **Settings → MAAP Settings → `maapToken`** (your `MAAP_PGT`), or
  `export MAAP_PGT=...` before launching the kernel.

Full write-up: [`docs/DPS.md` → Deleting (undeploying) an algorithm](../docs/DPS.md#deleting-undeploying-an-algorithm).

## 1. Connect

In [ ]:
import json
import requests
from maap.maap import MAAP

# OGC processes API.
# NOTE the /api suffix. This is NOT the same string as the JupyterLab extensions'
# `maapApiUrl` setting, which must have NO /api suffix (the extensions append it).
BASE = "https://api.maap-project.org/api/ogc/processes"

maap = MAAP()
headers = maap._get_api_header()   # private helper: carries `token` (+ MAAP_PGT proxy-ticket)

assert headers.get("token"), (
    "No MAAP token. Set Settings -> MAAP Settings -> maapToken (your MAAP_PGT), "
    "or export MAAP_PGT before starting the kernel."
)
print("auth header fields:", sorted(headers))

## 2. List every registered process

`processID` is the delete key. Note it for the ones you want gone.

In [ ]:
def fetch_processes():
    r = requests.get(BASE, headers=headers)
    r.raise_for_status()
    return r.json()["processes"]


def show(rows):
    """Print processID / name:version / deployer / last modified, sorted by ID."""
    if not rows:
        print("(none)")
        return
    for p in sorted(rows, key=lambda x: int(x["processID"])):
        name = f'{p["id"]}:{p["version"]}'
        print(f'processID={str(p["processID"]):<5} {name:<42} '
              f'deployedBy={str(p["deployedBy"]):<16} {p.get("lastModifiedTime", "")}')


procs = fetch_processes()
print(f"{len(procs)} processes registered\n")
show(procs)

In [ ]:
# Full shape of one entry, if you need a field the table above doesn't show.
print("KEYS:", list(procs[0].keys()), "\n")
print(json.dumps(procs[0], indent=2, default=str))

## 3. Narrow it down to yours

Set `ME` to your MAAP username (the `deployedBy` value in the table above) and/or
`NAME_CONTAINS` to a substring of the process id. Both blank = no filtering.

In [ ]:
ME = ""                       # e.g. "kdl0040" — blank = don't filter by deployer
NAME_CONTAINS = "disasters"   # e.g. "ogc-test", "umbra" — blank = don't filter by name

mine = [
    p for p in procs
    if (not ME or p["deployedBy"] == ME)
    and (not NAME_CONTAINS or NAME_CONTAINS in p["id"])
]
print(f"{len(mine)} of {len(procs)} match (ME={ME!r}, NAME_CONTAINS={NAME_CONTAINS!r})\n")
show(mine)

## 4. Delete

Put the `processID` **ints** in `DELETE_IDS`, run once with `DRY_RUN = True` to confirm
the targets, then flip it to `False`.

Response codes: **200** deleted · **403** you are not the deployer · **404** no such
`processID` (already gone, or you used the name instead of the ID).

In [ ]:
DELETE_IDS = []     # e.g. [30, 31] — processID ints from the tables above
DRY_RUN = True      # flip to False to actually delete

by_id = {str(p["processID"]): p for p in procs}

for pid in DELETE_IDS:
    p = by_id.get(str(pid))
    if p is None:
        print(f"{pid} -> not in the current listing; refusing (re-run step 2)")
        continue
    label = f'{pid} {p["id"]}:{p["version"]} (deployedBy={p["deployedBy"]})'
    if DRY_RUN:
        print("DRY RUN would delete:", label)
        continue
    r = requests.delete(f"{BASE}/{pid}", headers=headers)
    print(label, "->", r.status_code, r.text)
    # 200 = undeployed | 403 = not the deployer | 404 = no such processID

## 5. Verify

Re-list and diff against the pre-delete snapshot. Also confirm in the MAAP
**Submit Jobs → Process** dropdown — that dropdown is the real "is it deployed" check.

In [ ]:
after = fetch_processes()
removed = {str(p["processID"]) for p in procs} - {str(p["processID"]) for p in after}

print(f"{len(procs)} -> {len(after)} processes")
print("removed processIDs:", sorted(removed) or "none")
print()
show(after)

procs = after   # refresh the snapshot so step 4 can be re-run against current state

## Notes

- **Delete is per-`processID`, so a name with several versions has several IDs.** Deleting
  `foo:dev` leaves `foo:v1.2.0` alone.
- **A rename is not a move.** Editing `algorithm_name` in `algorithm_config.yaml` /
  `dps/ogc/<name>.yml` and re-registering creates a second process; the old name keeps
  running the code it was built with until you delete it here.
- **`403` on your own algorithm** usually means the token belongs to a different MAAP
  account than the one that registered it (e.g. registered from the ADE, deleting from
  the Disasters hub). Check `deployedBy`.
- **Deleting does not cancel running jobs** and does not remove outputs from
  `nasa-disasters-staging/dps_output/<activation_event>/`.